# 🎙️ Kaggle Speech AI Studio (SOTA STT & TTS)
### Bộ dịch vụ Âm thanh & Giọng nói AI chuyên biệt:
- 🎙️ **STT**: Qwen3-ASR 1.7B Multilingual (Tối ưu đặc biệt tiếng Việt & 50+ ngôn ngữ)
- 🔊 **TTS**: Qwen3-TTS 1.7B (Đa ngôn ngữ, VoiceDesign, CustomVoice)
- 🗣️ **XTTS-v2**: XTTS-v2 Vietnamese (Giọng đọc Tiếng Việt chuyên biệt & Voice Cloning)
- 🌐 **Chuẩn REST API**: OpenAI Compatible ,  qua Cloudflare Tunnel


In [ ]:
# 1. Nạp biến môi trường tự động (Hỗ trợ Kaggle Secrets, file .env, hoặc cấu hình runtime)
import os, json

# Thiết lập tối ưu GPU & Cache
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["IDLE_SHUTDOWN_MINUTES"] = "120"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["STUDIO_MODE"] = "speech"
os.environ["HF_HOME"] = "/tmp/hf_cache"

# Nạp Kaggle UserSecrets nếu được cấp quyền
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    for key in ["KAGGLE_USERNAME", "KAGGLE_KEY", "HF_TOKEN", "NTFY_TOPIC"]:
        val = secrets.get_secret(key)
        if val:
            os.environ[key] = val
except Exception:
    pass

# Đọc file .env nếu có sẵn
for env_candidate in [".env", "/kaggle/working/Gen_Image-Video/kaggle/all-in-one/.env"]:
    if os.path.exists(env_candidate):
        try:
            with open(env_candidate, "r") as ef:
                for line in ef:
                    line = line.strip()
                    if line and not line.startswith("#") and "=" in line:
                        k, v = line.split("=", 1)
                        clean_val = v.strip().strip("\"").strip("\x27")
                        os.environ.setdefault(k.strip(), clean_val)
        except Exception:
            pass

# Tạo cấu hình ~/.kaggle/kaggle.json để Kaggle API / CLI hoạt động thông suốt
try:
    if os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"):
        os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
        with open(os.path.expanduser("~/.kaggle/kaggle.json"), "w") as kf:
            json.dump({
                "username": os.environ["KAGGLE_USERNAME"],
                "key": os.environ["KAGGLE_KEY"]
            }, kf)
        os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
except Exception:
    pass

print(f"✅ Môi trường cấu hình hoàn tất! User: {os.environ.get('KAGGLE_USERNAME', 'Chưa cấu hình')}")


In [ ]:
# 2. Đồng bộ mã nguồn Studio AI (Live Payload)
import os, base64, io, tarfile

target_dir = "/kaggle/working/Gen_Image-Video/kaggle/all-in-one"
os.makedirs(target_dir, exist_ok=True)
os.chdir(target_dir)

payload_data = "..."
raw_tar = base64.b64decode(payload_data)
with tarfile.open(fileobj=io.BytesIO(raw_tar), mode="r:gz") as tar:
    tar.extractall(".")

print("✅ Toàn bộ module Studio AI phiên bản 3.5 đã đồng bộ thành công!")

In [ ]:
# --- CELL 3: SETUP ENVIRONMENT (100% Offline via kaggle-speech-wheels + Fallback) ---
import glob, os, shutil, subprocess, sys, time, urllib.request, torch, torchaudio

def _ntfy_dbg(msg: str):
    try:
        u = os.environ.get('KAGGLE_USERNAME', 'cuongnguyen1802')
        req = urllib.request.Request(f'https://ntfy.sh/studio-ai-debug-{u}', data=msg.encode('utf-8'), method='POST')
        urllib.request.urlopen(req, timeout=4)
    except Exception:
        pass

t_setup_0 = time.time()
_ntfy_dbg('[Speech] Cell 3 starting environment setup...')
bundles = glob.glob('/kaggle/input/**/speech_runtime_bundle.tar.bin', recursive=True)
_ntfy_dbg(f'[Speech] Bundles found: {len(bundles)}')

if bundles:
    bundle_path = bundles[0]
    print(f'🚀 [Offline Mode] Đang giải nén gói Runtime & Wheels từ: {bundle_path}')
    rt_dir = '/tmp/speech_rt'
    shutil.rmtree(rt_dir, ignore_errors=True)
    os.makedirs(rt_dir, exist_ok=True)
    subprocess.run(['tar', '-xzf', bundle_path, '-C', rt_dir], check=True)

    print('📦 [1/2] Cài đặt offline các gói hệ thống (.deb: sox, espeak-ng, cloudflared)...')
    subprocess.run(f'dpkg -i --force-depends {rt_dir}/debs/*.deb >/dev/null 2>&1', shell=True, check=False)

    print('📦 [2/2] Cài đặt offline toàn bộ Python wheels (--no-index --no-deps)...')
    whls = sorted(glob.glob(f'{rt_dir}/wheels/*.whl'))
    for whl in whls:
        subprocess.run(
            f'{sys.executable} -m pip install -q --no-index --no-deps {whl}',
            shell=True,
            check=False,
        )
    shutil.rmtree(rt_dir, ignore_errors=True)
    msg = f'[Speech] Offline install ({len(whls)} wheels) done in {time.time() - t_setup_0:.1f}s!'
    print('✅', msg)
    _ntfy_dbg(msg)
else:
    print('⚠️ Không tìm thấy speech_runtime_bundle.tar.bin, chuyển sang fallback cài đặt online...')
    _ntfy_dbg('[Speech] Fallback online installation starting...')
    with open('/tmp/torch_constraints.txt', 'w') as cf:
        cf.write('torch==' + str(torch.__version__) + '\n')
        cf.write('torchaudio==' + str(torchaudio.__version__) + '\n')
    subprocess.run('apt-get update -qq && apt-get install -y -qq sox libsox-dev libsox-fmt-all espeak-ng', shell=True)
    subprocess.run('pip install -q -c /tmp/torch_constraints.txt fastapi uvicorn pyyaml soundfile librosa einops onnxruntime sox sentencepiece tiktoken nagisa soynlp jieba qwen-omni-utils', shell=True)
    subprocess.run('pip install -q --upgrade --no-deps qwen-asr qwen-tts', shell=True)
    subprocess.run('pip install -q --upgrade --no-deps coqpit-config coqui-tts-trainer ko-speech-tools monotonic-alignment-search pysbd num2words unidecode inflect anyascii pypinyin coqui-tts', shell=True)
    subprocess.run("pip install -q -c /tmp/torch_constraints.txt 'transformers==4.57.6' 'huggingface-hub>=0.34.0,<1.0' 'accelerate>=1.12.0'", shell=True)
    if not os.path.exists('/usr/local/bin/cloudflared'):
        subprocess.run('wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && dpkg -i cloudflared-linux-amd64.deb', shell=True)

try:
    import fastapi, uvicorn, starlette, yaml
except Exception as imp_err:
    print(f'📦 Bổ sung phụ trợ FastAPI ({imp_err})...')
    _ntfy_dbg(f'[Speech] Installing FastAPI deps: {imp_err}')
    subprocess.run(
        f'{sys.executable} -m pip install -q fastapi uvicorn pyyaml python-multipart requests psutil soundfile',
        shell=True,
        check=False,
    )

_ntfy_dbg(f'[Speech] Setup completed in {time.time() - t_setup_0:.1f}s!')


In [ ]:
# 4. Khởi động Studio (speech mode - Hỗ trợ Intentional Shutdown exit 99)
import os, subprocess, sys, time, urllib.request

def _ntfy_dbg(msg: str):
    try:
        u = os.environ.get('KAGGLE_USERNAME', 'cuongnguyen1802')
        req = urllib.request.Request(f'https://ntfy.sh/studio-ai-debug-{u}', data=msg.encode('utf-8'), method='POST')
        urllib.request.urlopen(req, timeout=4)
    except Exception:
        pass

os.environ['STUDIO_MODE'] = 'speech'
for attempt in range(3):
    t0 = time.time()
    _ntfy_dbg(f'[Speech] Cell 4 launching main.py (attempt {attempt+1})...')
    ret = subprocess.run([sys.executable, '-u', 'main.py', '--mode', 'speech'], cwd=os.getcwd())
    _ntfy_dbg(f'[Speech] main.py exited with code {ret.returncode} after {time.time() - t0:.1f}s')
    if ret.returncode in (0, 99):
        print(f'Clean exit requested ({ret.returncode}). Stopping notebook.')
        break
    if time.time() - t0 < 15:
        print(f'Fast crash (code {ret.returncode}). Exiting to preserve error log.')
        break
    print(f'Server exited with code {ret.returncode}, restarting in 3s...')
    time.sleep(3)
